In [0]:
%sql
USE CATALOG dbt_aws_environment;
CREATE SCHEMA IF NOT EXISTS bronze;
USE bronze;

In [0]:
from pyspark.sql.functions import *

catalog = "dbt_aws_environment"

def ingest_to_bronze(source_name):
    """
    Using DBFS paths to Autoload data files into bronze tables.
    """

    checkpoint_path = f"/Workspace/Users/jamestc1439@gmail.com/.checkpoints/dbt_aws_environment/{source_name}"
    schema_path = f"/Workspace/Users/jamestc1439@gmail.com/.schemas/dbt_aws_environment/{source_name}"
    target_table = f"{catalog}.bronze.raw_{source_name}"

    print(f"  Ingesting: {source_name} → {target_table}")

    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("multiLine", "true")
            .option("escape", '"')
            .option("rescuedDataColumn", "_rescued_data")
            .option("cloudFiles.schemaLocation", schema_path)
            .option("pathGlobFilter", f"{source_name}.csv")
            .load("/Volumes/dbt_aws_environment/bronze/raw_files")
            .withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", col("_metadata.file_path"))
        .writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(target_table)
    )

In [0]:
sources = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv"
]

# Drop existing tables and clear checkpoints to avoid LOCATION_OVERLAP
for source in sources:
    source_base = source.replace(".csv", "")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.raw_{source_base}")
    dbutils.fs.rm(f"/Workspace/Users/jamestc1439@gmail.com/.checkpoints/dbt_aws_environment/{source_base}", recurse=True)
    dbutils.fs.rm(f"/Workspace/Users/jamestc1439@gmail.com/.schemas/dbt_aws_environment/{source_base}", recurse=True)

queries = []
for source in sources:
    source_base = source.replace(".csv", "")
    q = ingest_to_bronze(source_base)
    queries.append(q)

# Block until all streams complete
for q in queries:
    q.awaitTermination()

print("\nAll Bronze tables loaded into olist.bronze.")

In [0]:
print(f"{'Table':<40} {'Rows':>12}")
print("-" * 54)
for source in sources:
    source_base = source.replace(".csv", "")
    count = spark.table(f"{catalog}.bronze.raw_{source_base}").count()
    print(f"{catalog}.bronze.raw_{source_base:<24} {count:>12,}")

In [0]:
print("\n=== Rescued data check ===")
for source in sources:
    source_base = source.replace(".csv", "")
    rescued = (
        spark.table(f"{catalog}.bronze.raw_{source_base}")
             .filter("_rescued_data is not null")
             .count()
    )
    status = f"⚠️  {rescued} bad rows" if rescued > 0 else "✅  clean"
    print(f"  {catalog}.bronze.raw_{source_base:<24} {status}")